### Hamburg traffic data

Hamburg traffic data are available at varying levels of aggregation for different time periods. On the official website of the [City of Hamburg](https://www.hamburg.de/politik-und-verwaltung/behoerden/bvm/verkehrsstaerken-kfz-193324), one may find all related information. For older data, only weekly traffic counts are accessible through the [metaver portal](https://metaver.de/trefferanzeige?docuuid=2936465E-C045-4F5D-8614-24C3FBB522E2), but for more recent years data are provided on a daily or even hourly basis. These data can be downloaded on a JSON format, or queried as in the code below. The description in the metaver portal provide all necessary information to query the API.  

Below we obtain **daily** data (_Anzahl_Kfz_Zaehlstelle_1-Tag_), from **November 2021** to **March 2022**.

Data downloaded on: 2025-07-04

In [2]:
import requests
import pandas as pd
import os
from tqdm import tqdm
from shapely.geometry import Point
import geopandas as gpd
from datetime import datetime

In [ ]:
# output directory
path = "../../data-hamburg/01_raw/hamburg_traffic_data"
os.makedirs(path, exist_ok=True)

BASE_URL = "https://iot.hamburg.de/v1.1"
LAYER_NAME = "Anzahl_Kfz_Zaehlstelle_1-Tag" # weekly: "Anzahl_Kfz_Zaehlstelle_1-Woche", daily: "Anzahl_Kfz_Zaehlstelle_1-Tag", hourly: "Anzahl_Kfz_Zaehlstelle_1-Stunde"
SERVICE_NAME = "HH_STA_AutomatisierteVerkehrsmengenerfassung"
START_DATE = "2021-11-01T00:00:00Z"
END_DATE = "2022-03-31T23:59:59Z"

def get_thing_info(datastream_id):
    '''Fetch Thing info linked to Datastream'''
    url = f"{BASE_URL}/Datastreams({datastream_id})/Thing"
    resp = requests.get(url)
    resp.raise_for_status()
    thing = resp.json()
    return thing

def get_location_info(thing_id):
    '''Fetch Locations linked to Thing'''
    url = f"{BASE_URL}/Things({thing_id})/Locations"
    resp = requests.get(url)
    resp.raise_for_status()
    locations = resp.json()["value"]
    
    if len(locations) > 1:
            tqdm.write(f"Ambiguous location: {len(locations)} available for ID {thing_id}.")

    if locations:
        loc = locations[0]
        return loc.get("name"), Point(*loc.get("location").get("geometry").get("coordinates"))
    
    return None, None

def fetch_observations(datastream_id):
    '''Fetch observations with pagination'''
    obs_url = f"{BASE_URL}/Datastreams({datastream_id})/Observations"
    obs_params = {
        "$filter": f"phenomenonTime ge {START_DATE} and phenomenonTime le {END_DATE}",
        "$top": 1000
    }

    observations = []
    while obs_url:
        resp = requests.get(obs_url, params=obs_params)
        resp.raise_for_status()
        data = resp.json()
        for obs in data["value"]:
            phenomenonTime = obs["phenomenonTime"].split('/')
            start_time = datetime.fromisoformat(phenomenonTime[0].replace("Z", "+00:00"))
            end_time = datetime.fromisoformat(phenomenonTime[1].replace("Z", "+00:00"))

            observations.append({
                "phenomenonTime": obs["phenomenonTime"],
                "start_time": start_time,
                "end_time" : end_time,
                "traffic_count": obs["result"]
            })
        obs_url = data.get("@iot.nextLink")
        obs_params = None # remove after first request
    return observations

# Fetch Datastreams
ds_url = f"{BASE_URL}/Datastreams"
ds_params = {
    "$filter": f"properties/serviceName eq '{SERVICE_NAME}' and properties/layerName eq '{LAYER_NAME}'",
    "$top": 1000
}

print("Fetching all Datastreams...")
ds_resp = requests.get(ds_url, params=ds_params)
ds_resp.raise_for_status()
datastreams = ds_resp.json()["value"]

print(f"Found {len(datastreams)} datastreams.")

data_list = []

for ds in tqdm(datastreams, desc="Processing datastreams"):
    ds_id = ds["@iot.id"]
    ds_name = ds.get("name", f"Datastream_{ds_id}").replace("/", "_")

    # get sensor site (Thing) info
    thing = get_thing_info(ds_id)
    thing_id = thing.get("@iot.id")
    thing_name = thing.get("name", f"Thing_{thing_id}")

    # get location
    loc_name, loc_geom = get_location_info(thing_id)

    # fetch observations
    observations = fetch_observations(ds_id)

    # create df for site
    df = pd.DataFrame(observations)
    df["datastream_id"] = ds_id
    df["datastream_name"] = ds_name
    df["thing_id"] = thing_id
    df["thing_name"] = thing_name
    df["location_name"] = loc_name
    df["geometry"] = loc_geom 

    # store df
    data_list.append(df)


# single df
hamburg_traffic_df = pd.concat(data_list, ignore_index=True)
hamburg_traffic_gdf = gpd.GeoDataFrame(hamburg_traffic_df, geometry="geometry", crs="EPSG:4326")
hamburg_traffic_gdf.to_parquet(f"{path}/hamburg_traffic.parquet")

Fetching all Datastreams...
Found 815 datastreams.


Processing datastreams: 100%|██████████| 815/815 [05:06<00:00,  2.66it/s]


In [4]:
hamburg_traffic_gdf.head()

,phenomenonTime,start_time,end_time,traffic_count,datastream_id,datastream_name,thing_id,thing_name,location_name,geometry
0,2022-03-30T22:00:00Z/2022-03-31T21:59:59Z,2022-03-30 22:00:00+00:00,2022-03-31 21:59:59+00:00,7674.0,12960,Kfz-Aufkommen an Verkehrszählstelle 0332930 im...,5919,Verkehrszählstelle 0332930,Verkehrszählstelle 0332930,POINT (10.00181 53.55659)
1,2022-03-24T23:00:00Z/2022-03-25T22:59:59Z,2022-03-24 23:00:00+00:00,2022-03-25 22:59:59+00:00,8192.0,12960,Kfz-Aufkommen an Verkehrszählstelle 0332930 im...,5919,Verkehrszählstelle 0332930,Verkehrszählstelle 0332930,POINT (10.00181 53.55659)
2,2022-03-23T23:00:00Z/2022-03-24T22:59:59Z,2022-03-23 23:00:00+00:00,2022-03-24 22:59:59+00:00,8215.0,12960,Kfz-Aufkommen an Verkehrszählstelle 0332930 im...,5919,Verkehrszählstelle 0332930,Verkehrszählstelle 0332930,POINT (10.00181 53.55659)
3,2022-03-22T23:00:00Z/2022-03-23T22:59:59Z,2022-03-22 23:00:00+00:00,2022-03-23 22:59:59+00:00,8351.0,12960,Kfz-Aufkommen an Verkehrszählstelle 0332930 im...,5919,Verkehrszählstelle 0332930,Verkehrszählstelle 0332930,POINT (10.00181 53.55659)
4,2022-03-21T23:00:00Z/2022-03-22T22:59:59Z,2022-03-21 23:00:00+00:00,2022-03-22 22:59:59+00:00,8153.0,12960,Kfz-Aufkommen an Verkehrszählstelle 0332930 im...,5919,Verkehrszählstelle 0332930,Verkehrszählstelle 0332930,POINT (10.00181 53.55659)


In [24]:
hamburg_traffic_gdf.geometry.nunique(), hamburg_traffic_gdf.thing_id.nunique()

(693, 693)